# 05 — Fetch Inactive Subscriptions

Separate, much simpler pipeline for subscriptions whose `SubscriptionEndDate`
is already in the past (split off from the active batch in
`05_Fetch_Subscriptions.ipynb`, step 2b).

**No Voyager lookup** — the circuits endpoint
(`GET /fibre/v1/circuits/{SupplierServiceID}`) doesn't return anything for a
subscription that's already ended, so there's nothing to look up. Instead:

- **Radius Username** = the vBill `SubscriptionLabel`, used as-is.
- **Address** — not looked up or created here at all. These subscriptions
  attach to whichever address is already the account's default service
  address — see `06_Attach_Inactive_Addresses.ipynb`.

Target account resolution (own account / Managed by Williams / Williams
Corporation) uses the exact same logic as the active pipeline.


## 1. Setup


In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

logger = get_logger("fetch_inactive_subscriptions")

df_subscriptions = try_load_df("subscriptions_inactive_raw", dtype={"AccountCode": str, "SupplierServiceID": str})
if df_subscriptions is None or df_subscriptions.empty:
    logger.warning(
        "No 'subscriptions_inactive_raw' file found (or it's empty) — run "
        "05_Fetch_Subscriptions.ipynb first (step 2b saves this file), or there "
        "simply were no inactive subscriptions in that run."
    )
    df_subscriptions = pd.DataFrame()
else:
    logger.info(f"Loaded {len(df_subscriptions):,} inactive subscriptions")

df_subscriptions.head()


2026-07-24 14:46:46,386 [INFO] Loaded 59 inactive subscriptions


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,4170938331,2026-07-03 20:36:54,260909,vBill,99965692,Broadband - Fibre,V113062905,V113062905_0@no-username,2025-12-04,2026-06-20,...,UFB 100/20/2.5/2.5,NaN,WC CHCH T9,2026-03-19: RQ [CPP 1007277]; CPP 1006638,NaN,NaN,NaN,NaN,NaN,NaN
1,4170938476,2026-07-03 20:37:32,260908,vBill,99965692,Broadband - Fibre,V113062897,V113062897_0@williamsinternet.com,2025-12-03,2026-03-06,...,UFB 100/20/2.5/2.5,NaN,Managed by Williams Limited,2025-03-05 RQ [CPP 1006916]; CPP 1006643,NaN,NaN,NaN,NaN,NaN,NaN
2,4170938481,2026-07-03 20:36:54,260910,vBill,99965692,Broadband - Fibre,V113062913,V113062913_0@williamsinternet.com,2025-12-04,2026-06-12,...,UFB 100/20,NaN,WC CHCH T9,2026-06-12 RQ[CPP 1007254] ; CPP 1006639,NaN,NaN,NaN,NaN,NaN,NaN
3,4170938610,2026-07-03 20:37:23,260926,vBill,99965692,Broadband - Fibre,V113063077,V113063077_0@williamsinternet.com,2025-12-04,2026-03-11,...,UFB 100/20/2.5/2.5,NaN,WC CHCH T9,2026-10-03: RQ [CPP 3505039]; CPP 3501020,NaN,NaN,NaN,NaN,NaN,NaN
4,4170938660,2026-07-03 20:36:55,260944,vBill,99965692,Broadband - Fibre,V113063259,V113063259_0@no-username,2025-12-04,2026-06-12,...,UFB Max/500/2.5/2.5,NaN,WC CHCH T9,2026-06-11 RQ[\tCPP 3506360\t ] ; CPP 3501043,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Resolve target account for every subscription

Identical logic to `05_Fetch_Subscriptions.ipynb` — Managed by Williams
`Reference` match wins, then own account, then Williams Corporation
fallback. See that notebook for the full explanation.


In [6]:
if df_subscriptions.empty:
    df_subscriptions["TargetAccountKey"] = pd.Series(dtype=str)
    df_subscriptions["TargetAccountNumber"] = pd.Series(dtype=str)
else:
    own_account_map = load_account_code_batch_map()  # {AccountCode: AccountCode_Batch}, every account in 04's results
    real_account_numbers = load_real_target_account_numbers()  # {"managed_by_williams": "...", "williams_corporation": "..."}

    missing_keys = set(TARGET_ACCOUNTS) - set(real_account_numbers)
    if missing_keys:
        logger.warning(
            f"No successful account_results row found for: {missing_keys} — "
            f"falling back to the TARGET_ACCOUNTS placeholder for those. "
            f"Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run."
        )

    reference_col = SUBSCRIPTION_REFERENCE_COLUMN if SUBSCRIPTION_REFERENCE_COLUMN in df_subscriptions.columns else None
    if reference_col is None:
        logger.warning(f"Column '{SUBSCRIPTION_REFERENCE_COLUMN}' not found in df_subscriptions — Managed-by-Williams routing can't work without it.")
        df_subscriptions["CustomerSuppliedReference"] = None
    else:
        df_subscriptions["CustomerSuppliedReference"] = df_subscriptions[reference_col]

    df_subscriptions["AccountCode"] = df_subscriptions["AccountCode"].astype(str)

    is_managed_by_williams = (
        df_subscriptions["CustomerSuppliedReference"].fillna("").astype(str).str.lower()
        .str.contains(MANAGED_BY_WILLIAMS_MARKER, regex=False)
    )

    managed_by_williams_number = real_account_numbers.get(
        "managed_by_williams", TARGET_ACCOUNTS["managed_by_williams"]["account_number"]
    )
    williams_corporation_number = real_account_numbers.get(
        "williams_corporation", TARGET_ACCOUNTS["williams_corporation"]["account_number"]
    )

    df_subscriptions["TargetAccountKey"] = None
    df_subscriptions["TargetAccountNumber"] = None

    # 1. Managed by Williams — always wins, regardless of own account.
    df_subscriptions.loc[is_managed_by_williams, "TargetAccountKey"] = "managed_by_williams"
    df_subscriptions.loc[is_managed_by_williams, "TargetAccountNumber"] = managed_by_williams_number

    # 2. Everyone else — own account first.
    not_managed = ~is_managed_by_williams
    df_subscriptions.loc[not_managed, "TargetAccountNumber"] = df_subscriptions.loc[not_managed, "AccountCode"].map(own_account_map)
    df_subscriptions.loc[not_managed & df_subscriptions["TargetAccountNumber"].notna(), "TargetAccountKey"] = "own_account"

    # 3. Everyone else, still without an account — Williams Corporation bucket fallback.
    missing_own_account = not_managed & df_subscriptions["TargetAccountNumber"].isna()
    if missing_own_account.any():
        df_subscriptions.loc[missing_own_account, "TargetAccountKey"] = "williams_corporation"
        df_subscriptions.loc[missing_own_account, "TargetAccountNumber"] = williams_corporation_number

    logger.info(df_subscriptions["TargetAccountKey"].value_counts(dropna=False).to_string())

df_subscriptions[["SubscriptionUSN", "AccountCode", "TargetAccountKey", "TargetAccountNumber"]].head(20)


2026-07-24 14:46:46,625 [WARNING] No successful account_results row found for: {'williams_corporation'} — falling back to the TARGET_ACCOUNTS placeholder for those. Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run.
2026-07-24 14:46:46,675 [INFO] TargetAccountKey
own_account            36
managed_by_williams    23


,SubscriptionUSN,AccountCode,TargetAccountKey,TargetAccountNumber
0,V113062905,99965692,own_account,99965692_fullbatch4
1,V113062897,99965692,managed_by_williams,999656921_fullbatch4
2,V113062913,99965692,own_account,99965692_fullbatch4
3,V113063077,99965692,own_account,99965692_fullbatch4
4,V113063259,99965692,own_account,99965692_fullbatch4
5,V113063267,99965692,own_account,99965692_fullbatch4
6,V113064125,99965692,own_account,99965692_fullbatch4
7,V113064349,99965692,own_account,99965692_fullbatch4
8,V113064380,99965692,own_account,99965692_fullbatch4
9,V113064372,99965692,own_account,99965692_fullbatch4


## 3. Radius Username — SubscriptionLabel, no lookup

No API call. `ParsedAddress_radius_user` is set straight from `SubscriptionLabel`
so `07_Create_Inactive_Subscription_Orders.ipynb` (which shares
`build_subscription_order_payload` with the active pipeline) doesn't need any
special-casing — it already falls back to `SubscriptionLabel` whenever
`ParsedAddress_radius_user` isn't set from Voyager.

The other `ParsedAddress_*` columns (addLine1, city, etc.) are created here
too, all `None` — so this dataframe has the same shape as the active
pipeline's, even though no address was looked up.


In [7]:
df_subscriptions["ParsedAddress_radius_user"] = df_subscriptions["SubscriptionLabel"]
for _col in ["addLine1", "addLine2", "city", "postcode", "region_name", "region_iso", "region_code_raw", "location_id"]:
    df_subscriptions[f"ParsedAddress_{_col}"] = None

df_subscriptions["ParsedAddress_parsed_ok"] = df_subscriptions["ParsedAddress_radius_user"].apply(lambda v: clean(v) is not None)
df_subscriptions["ParsedAddress_error"] = df_subscriptions["ParsedAddress_parsed_ok"].map(
    {True: None, False: "SubscriptionLabel is blank — no Radius Username available"}
)

_unresolved = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not _unresolved.empty:
    logger.warning(f"{len(_unresolved):,} inactive subscriptions have a blank SubscriptionLabel — no Radius Username available")

df_subscriptions[["SubscriptionUSN", "SubscriptionLabel", "ParsedAddress_radius_user", "ParsedAddress_parsed_ok"]].head(20)


,SubscriptionUSN,SubscriptionLabel,ParsedAddress_radius_user,ParsedAddress_parsed_ok
0,V113062905,V113062905_0@no-username,V113062905_0@no-username,True
1,V113062897,V113062897_0@williamsinternet.com,V113062897_0@williamsinternet.com,True
2,V113062913,V113062913_0@williamsinternet.com,V113062913_0@williamsinternet.com,True
3,V113063077,V113063077_0@williamsinternet.com,V113063077_0@williamsinternet.com,True
4,V113063259,V113063259_0@no-username,V113063259_0@no-username,True
5,V113063267,V113063267_0@williamsinternet.com,V113063267_0@williamsinternet.com,True
6,V113064125,V113064125_0@williamsinternet.com,V113064125_0@williamsinternet.com,True
7,V113064349,V113064349_0@williamsinternet.com,V113064349_0@williamsinternet.com,True
8,V113064380,V113064380_0@williamsinternet.com,V113064380_0@williamsinternet.com,True
9,V113064372,V113064372_0@williamsinternet.com,V113064372_0@williamsinternet.com,True


## 4. Save


In [8]:
save_df("subscriptions_inactive_resolved", df_subscriptions)


Saved 59 rows -> migration_data\05_subscriptions_inactive_resolved.csv
